# Continuous time models

## Brownian motion

In [ ]:
import numpy as np
import pandas as pd
import random

from scipy.stats import norm

import plotly.graph_objects as go
from scipy.stats.contingency import crosstab
from scipy.stats import chi2_contingency

In [ ]:
def bm_simulations(n_paths, granularity, max_time):
    n_points = granularity * max_time
    
    paths = []
    for _ in range(n_paths):
        xs = [0] + random.choices([-1, 1], weights = [0.5, 0.5], k = n_points)
        path = np.cumsum(xs)/np.sqrt(granularity)
        paths.append(path)
    
    return np.array(paths)

In [ ]:
T = 1
n = 1000
N = 3

t = np.linspace(0, T, n*T + 1)
paths = bm_simulations(N, n, T)

In [ ]:
plot = go.Figure()

for i in range(0,N):
    graph = go.Scatter(x = t, y = paths[i])
    plot.add_trace(graph)
    
plot.show()

Check expectation and variance of $W_{0.6}$

In [ ]:
T = 1
n = 100
N = 25000

paths = bm_simulations(N, n, T)
W_06 = paths[:, 60] 

In [ ]:
print('Expectation', np.mean(W_06), abs(np.mean(W_06) - 0))
sample_var = np.std(W_06, ddof = 1)**2
print('Variance', sample_var, abs(sample_var - 0.6))

$Cov(W_{0.4}, W_{0.6})$

In [ ]:
W_04 = paths[:, 40]
sample_cov = np.cov(W_04, W_06)[0][1]
print('Covariance', sample_cov, abs(sample_cov - min(0.4, 0.6)))

Check *independent increments* property. For example, use 2-way $\chi^2$ test to check that $X=(W_1-W_{0.6})$ and $Y=W_{0.6}-W_0 = W_{0.6}$ are independent.

Step 1. Divide $(-\infty, +\infty)$ into 3 categories: $(-\infty, -1],\,(-1,1),\,[1, +\infty)$ and count number of occurences of random variables in each of these categories.

In [ ]:
T = 1
n = 100
N = 100_000

paths = bm_simulations(N, n, T)

In [ ]:
Y = paths[:, 60]
X = paths[:, -1] - paths[:, 60]

In [ ]:
x1 = sum(X <=- 1)
x2 = sum((X<1) & (X>-1))
x3 = sum(X>=1)
count_X = [x1, x2, x3]

In [ ]:
X_categories = []
for x in X:
    if x <= -1:
        X_categories.append(1)
    if x > -1 and x < 1:
        X_categories.append(2)
    if x >= 1:
        X_categories.append(3)
        
Y_categories = []
for y in Y:
    if y <= -1:
        Y_categories.append(1)
    if y > -1 and y < 1:
        Y_categories.append(2)
    if y >= 1:
        Y_categories.append(3)

In [ ]:
# print(X_categories, Y_categories)

Step 2. Compute contingency table with observable values.<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.crosstab.html#scipy.stats.contingency.crosstab

In [ ]:
cont_table = crosstab(X_categories, Y_categories)

In [ ]:
cont_table[1]

Step 3. Apply $\chi^2$ test for $H_0: (W_1-W_{0.6}) \perp \!\!\! \perp W_{0.6}$.<br>
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html

In [ ]:
ans = chi2_contingency(cont_table[1])

In [ ]:
ans

In [ ]:
# help(chi2_contingency)

### Stochastic differential equations

Simulate Vasicek model
$dr_t = (\alpha - \beta r_t) dt + \sigma dW_t$
for arbitrary $\alpha, \beta, \sigma$

In [ ]:
alpha = 0.05
beta = 0.03
sigma = 0.1
r0 = 2
T = 1

m = 1000
h = 1/m
N = 5

paths = []
for _ in range(N):
    ksis = norm.rvs(0, np.sqrt(h), size = m*T)
    path = [r0]
    for i in range(1, m*T+1):
        next_pt = path[-1] + (alpha-beta*path[-1])*h +sigma*ksis[i-1]
        path.append(next_pt)
    paths.append(np.array(path))

In [ ]:
times = np.linspace(0, T, m*T+1) 

plot = go.Figure()
for path in paths:
    graph = go.Scatter(x = times, y = path)
    plot.add_trace(graph)
plot.show()